Snowflake Access Control notes covering roles, primary/secondary roles, and best practices

*Co-authored with CoCo*

## Access Control Framework

Snowflake's approach to access control combines aspects from the following models:

| Model | Description |
|-------|-------------|
| **Discretionary Access Control (DAC)** | Each object has an owner, who can in turn grant access to that object |
| **Role-based Access Control (RBAC)** | Access privileges are assigned to roles, which are in turn assigned to users |
| **User-based Access Control (UBAC)** | Access privileges are assigned directly to users. Considered only when `USE SECONDARY ROLE` is set to `ALL` |

---

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Securable Object** | An entity to which access can be granted. Unless allowed by a grant, access is denied |
| **Role** | An entity to which privileges can be granted or revoked |
| **Privilege** | A defined level of access to an object. Multiple distinct privileges control the granularity of access granted |
| **User** | A user identity recognized by Snowflake, whether associated with a person or service. A user is also an entity to which privileges can be granted |

> In Snowflake, privileges assigned to roles or users allow access to securable objects. Roles can be assigned to users or other roles. Granting a role to another role creates a **role hierarchy**.

---

### 1. Securable Object

Any object whose access can be controlled or managed (through granting/revoking privileges) is a securable object in Snowflake.

- By default, **nobody** can access a securable object unless the required privilege is granted
- Every securable object has an **owner** (a role) that has full control over it
- Privileges (`SELECT`, `INSERT`, `USAGE`, etc.) are granted **on** securable objects **to** roles

#### Examples of Securable Objects

| Category | Securable Objects |
|----------|-------------------|
| **Account-level** | Database, Warehouse, User, Role, Integration |
| **Database-level** | Schema, Database Role |
| **Schema-level** | Table, View, Stage, Stream, Task, Procedure, Function, Sequence, Pipe |

![Snowflake Securable Objects Hierarchy Image](snow://workspace/USER$.PUBLIC.DEFAULT$/versions/head/securable_objects_hierarchy.png "Snowflake Securable Objects")

#### Ownership

Any role that has the `OWNERSHIP` privilege on an object, owns the object. Each securable object is owned by a single role, which by default is the role used to create the object.

- When this role is assigned to users, they effectively have **shared control** over the object
- Using the owning role as primary role, a user can execute `GRANT OWNERSHIP` to transfer ownership to another role (including database roles)

**Who can transfer ownership:**
- The role that currently owns the object
- Any parent role in the hierarchy above the owning role (due to inheritance)
- `ACCOUNTADMIN` can always transfer ownership

---

### 2. Types of Roles

| Role Type | Scope | Key Characteristic |
|-----------|-------|--------------------|
| **Account Roles** | Entire account | Can be activated in a session |
| **Database Roles** | Single database | Cannot be activated directly in a session |
| **Application Roles** | Native App | Created by provider in setup script |

#### Account Roles
Used to permit SQL actions on any object in your account. Allows granting privileges on objects to an account role.

#### Database Roles
Used to limit SQL actions to a single database and its objects. Allows granting privileges on objects to a database role in the same database.

> **Note:** Database roles cannot be activated directly in a session. If you intend the user to activate a role in a session, grant database roles to account roles.

#### Application Roles
To enable consumer access to objects in a Snowflake Native App, the provider creates the application role and grants privileges to it in the setup script.

---

### 3. Active Roles

An **active role** is any role whose privileges are currently in effect during a session. Both the primary role and any secondary roles can be activated.

A role becomes active in either of the following ways:

1. When a session is first established, the user's **default role** and **default secondary roles** are activated
2. Executing `USE ROLE` or `USE SECONDARY ROLES` activates a different primary/secondary role

> **Note:** If a role is granted to a user but not activated, it's an **inactive/available role** — the user has it, but its privileges aren't being used.

```sql
-- Check which roles are currently active
SELECT CURRENT_ROLE();            -- primary active role
SELECT CURRENT_SECONDARY_ROLES(); -- secondary active roles

-- Check if a specific role is active in the session
SELECT IS_ROLE_IN_SESSION('ANALYST');  -- returns TRUE/FALSE
```

---

### 4. System-Defined Roles

| Role | Description |
|------|-------------|
| **GLOBALORGADMIN** | Performs organization-level tasks (managing accounts, org-level usage). Exists only in the organization account |
| **ORGADMIN** | Uses a regular account for org-level operations. Being phased out in favor of GLOBALORGADMIN |
| **ACCOUNTADMIN** | Top-level role in the account. Encapsulates SYSADMIN + SECURITYADMIN. Should be granted to very few users |
| **SECURITYADMIN** | Manages any object grant globally, creates/monitors/manages users and roles. Has `MANAGE GRANTS` privilege. Inherits USERADMIN |
| **USERADMIN** | Dedicated to user and role management. Has `CREATE USER` and `CREATE ROLE` privileges |
| **SYSADMIN** | Creates warehouses, databases, and other objects. Central administration role for non-security objects |
| **PUBLIC** | Pseudo-role automatically granted to every user and role. Objects owned by PUBLIC are available to everyone |

#### SECURITYADMIN Details

- Granted the `MANAGE GRANTS` privilege — can modify or revoke any grant
- `MANAGE GRANTS` does **not** allow creating objects — additional privileges are needed for that
- Inherits USERADMIN via the system role hierarchy

#### SYSADMIN and the Recommended Role Hierarchy

SYSADMIN has built-in privileges to create account-level objects (warehouses, databases, etc.).

**Snowflake recommends** that all custom roles should ultimately roll up to SYSADMIN:

```
ACCOUNTADMIN
  └── SYSADMIN
        ├── ANALYST_ROLE (custom)
        ├── DEV_ROLE (custom)
        └── DATA_ENGINEER_ROLE (custom)
```

```sql
-- Link custom roles to SYSADMIN
GRANT ROLE ANALYST_ROLE TO ROLE SYSADMIN;
GRANT ROLE DEV_ROLE TO ROLE SYSADMIN;
GRANT ROLE DATA_ENGINEER_ROLE TO ROLE SYSADMIN;
```

**Why this is important:**

Because of role inheritance, if `ANALYST_ROLE` owns a table, then SYSADMIN (as a parent) **inherits** all privileges of `ANALYST_ROLE` — including ownership. This means SYSADMIN can:
- Manage all objects created by any custom role beneath it
- Grant privileges on those objects to other roles
- Act as a central administration point for all non-security objects

#### What Happens with Orphaned Custom Roles

If a custom role is **not** granted to SYSADMIN (or any system role), it becomes "orphaned":

```
ACCOUNTADMIN
  └── SYSADMIN        ← cannot see LONE_ROLE's objects

LONE_ROLE             ← orphaned (not under SYSADMIN)
  └── owns TABLE_X
```

**Issues with orphaned roles:**
- SYSADMIN has **no access** to objects owned by the orphaned role
- Only `ACCOUNTADMIN` or the orphaned role itself can manage those objects
- Forces unnecessary use of ACCOUNTADMIN for routine tasks (violates least-privilege)
- Objects become harder to discover, audit, and administer
- If the role is dropped or no user has it, objects may become effectively inaccessible
- Creates gaps in centralized governance and access management

> **Best Practice:** Always link custom roles to SYSADMIN so there's a central role that can manage all objects without needing ACCOUNTADMIN for everyday operations.

---

### 5. Custom Roles

- **Custom account roles** can be created by USERADMIN (or higher) or any role with `CREATE ROLE` privilege
- **Custom database roles** can be created by the database owner (role with `OWNERSHIP` on the database)
- By default, a newly-created role is **not assigned to any user**, nor granted to any other role

> **Important:** Always grant custom roles to SYSADMIN to maintain a clean, manageable hierarchy.

![System Role Hierarchy Image](snow://workspace/USER$.PUBLIC.DEFAULT$/versions/head/system_role_hierarchy.png "System Role Hierarchy")

> **Note:** `ORGADMIN` is a separate system role that manages operations at the organization level. This role is not included in the hierarchy of system roles.

---

### 6. Users vs Roles — Who Actually Does Things?

In Snowflake, **users don't perform actions directly — roles do everything**. A user is just an identity that authenticates (logs in), but all actions are performed through the **active primary role**.

| Concept | Purpose |
|---------|--------|
| **User** | Identity / authentication (who you are) |
| **Role** | Authorization / privileges (what you can do) |

**How it works:**

1. A **user** logs in (authentication)
2. A **primary role** is activated for the session
3. Every action (`CREATE`, `SELECT`, `GRANT`, etc.) is executed **by the role**, not the user

```sql
-- The user types this, but USERADMIN (the active role) performs the action
USE ROLE USERADMIN;
CREATE ROLE ANALYST;  -- USERADMIN created and owns this role
```

**Key points:**
- Without an active primary role, a user **cannot** perform any action
- Every object is created by and owned by a **role**, never by a user directly
- When we say "a user creates a table," what actually happens is the user's **active primary role** creates and owns that table
- That's why a newly-created role is not assigned to any user — because the **creating role** owns it, but it remains isolated until explicitly granted to a user or another role

## Primary and Secondary Roles in Snowflake

### Overview

Snowflake uses a role-based access control (RBAC) model where every session has a **primary role** and optionally one or more **secondary roles**. Together they determine what a user can do within a session.

---

### Primary Role

The **primary role** is the main active role for a session. It determines:
- Object ownership for any objects created during the session
- The default privileges used to execute statements
- The role shown in query history and audit logs

#### How to Set / Activate a Primary Role

```sql
-- Set the primary role for the current session
USE ROLE <role_name>;

-- Example
USE ROLE SYSADMIN;
```

#### How to Create a Role (that can be used as primary)

```sql
-- Create a custom role
CREATE ROLE analyst_role;

-- Grant the role to a user
GRANT ROLE analyst_role TO USER john;

-- Grant privileges to the role
GRANT USAGE ON DATABASE analytics_db TO ROLE analyst_role;
GRANT USAGE ON SCHEMA analytics_db.public TO ROLE analyst_role;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics_db.public TO ROLE analyst_role;
```

#### Setting a Default Primary Role for a User

```sql
-- Set the default role that activates on login
ALTER USER john SET DEFAULT_ROLE = 'ANALYST_ROLE';
```

> **Note:** The user must already have the role granted to them.

---

### Secondary Roles

**Secondary roles** are additional roles activated alongside the primary role. They expand the session's privileges without changing object ownership.

- Objects created still belong to the **primary role**
- Privileges from all secondary roles are **unioned** with the primary role's privileges
- Useful when a user needs combined privileges from multiple roles without switching

#### How to Activate Secondary Roles

```sql
-- Activate ALL granted roles as secondary roles
USE SECONDARY ROLES ALL;

-- Activate specific roles as secondary roles (Snowflake 2024+)
USE SECONDARY ROLES <role1>, <role2>;

-- Deactivate all secondary roles
USE SECONDARY ROLES NONE;
```

#### How to Set a Default Secondary Role for a User

```sql
-- Set default secondary roles to ALL (activates on every login)
ALTER USER john SET DEFAULT_SECONDARY_ROLES = ('ALL');

-- Set default secondary roles to NONE
ALTER USER john SET DEFAULT_SECONDARY_ROLES = ('NONE');
```

#### How to Create and Assign a Secondary Role

Secondary roles are regular roles — any granted role can serve as a secondary role:

```sql
-- Create roles
CREATE ROLE data_reader;
CREATE ROLE data_writer;

-- Grant roles to a user
GRANT ROLE data_reader TO USER john;
GRANT ROLE data_writer TO USER john;

-- Now john can activate both as secondary roles
-- USE ROLE data_reader;           -- primary
-- USE SECONDARY ROLES data_writer; -- secondary
```

---

### Checking Current Roles

```sql
-- View the current primary role
SELECT CURRENT_ROLE();

-- View active secondary roles
SELECT CURRENT_SECONDARY_ROLES();

-- View all roles granted to the current user
SHOW GRANTS TO USER <username>;
```

---

### Key Differences: Primary vs Secondary Roles

| Aspect | Primary Role | Secondary Roles |
|--------|-------------|----------------|
| **Count per session** | Exactly 1 | 0 or more |
| **Object ownership** | Objects created are owned by primary role | Do NOT affect ownership |
| **Activation** | `USE ROLE <name>` | `USE SECONDARY ROLES ALL / <names>` |
| **Default setting** | `DEFAULT_ROLE` | `DEFAULT_SECONDARY_ROLES` |
| **Privilege scope** | Base privileges for session | Additive privileges combined with primary |
| **Audit/history** | Logged as the executing role | Not logged as the executing role |

---

### Limitations

1. **Object ownership:** Objects created in a session are always owned by the **primary role** — secondary roles cannot own objects.
2. **CREATE and ownership statements:**`CREATE <object>`, `GRANT OWNERSHIP` statements are executed **only** against the primary role, not secondary roles.
3. **Policy enforcement:** Row access policies and masking policies evaluate only the **primary role** via `CURRENT_ROLE()` — secondary roles are not considered unless the policy explicitly calls `IS_ROLE_IN_SESSION()`.
4. **Future grants:** Future grants apply based on the primary role context, not secondary roles.
5. **Sharing:** When accessing shared data (via data sharing / secure views), privilege checks use the primary role only.
6. **Role hierarchy:** Secondary roles include their own inherited/child roles, but the hierarchy operates independently from the primary role's hierarchy.

---

### Best Practices

- Use `DEFAULT_SECONDARY_ROLES = ('ALL')` for users who regularly need combined privileges
- Keep the primary role as the one that should own newly created objects
- Use `IS_ROLE_IN_SESSION('<role>')` in policies instead of `CURRENT_ROLE()` to account for secondary roles
- Follow least-privilege: only activate secondary roles when needed

### Role Hierarchy Explanation

In Snowflake, roles can be granted to other roles, forming a parent-child hierarchy. A parent role automatically inherits all privileges of its child roles.

When a secondary role is activated, it brings along its **entire sub-hierarchy of child roles** — just like a primary role does. However, the two hierarchies don't merge into one; they operate as **separate privilege sets**.

#### Example

```
ROLE_A (primary)
  └── ROLE_A1 (child of A)

ROLE_B (secondary)
  └── ROLE_B1 (child of B)
      └── ROLE_B2 (child of B1)
```

If `ROLE_A` is your primary role and `ROLE_B` is your secondary role:

- Your session gets privileges from: `ROLE_A` + `ROLE_A1` (primary hierarchy) **+** `ROLE_B` + `ROLE_B1` + `ROLE_B2` (secondary hierarchy)
- But `ROLE_A` does **not** inherit `ROLE_B`'s children, and `ROLE_B` does **not** inherit `ROLE_A`'s children
- They remain independent trees — their privileges are **unioned at the session level**, but the hierarchy chains themselves don't cross over

#### Why This Matters

If a policy or privilege check walks up the role hierarchy (e.g., checking ownership chains), it only walks **one tree at a time**. The secondary role's hierarchy won't be visible when evaluating the primary role's lineage, and vice versa.

For example, if `ROLE_B1` owns a table, `ROLE_B` (as a secondary role) can access it through inheritance. But `ROLE_A` (the primary role) cannot access that table through the secondary role's hierarchy — it only gains the **privileges**, not the **ownership chain**.

## Types of Custom Roles
Custom roles are of two types:
1. Account Role
2. Database Role

---

### 1. Account Role

Used to manage permissions across the **entire Snowflake account**.

Account roles can have privileges on: Warehouses, Databases, Schemas, Tables, Views, Stages, Pipes, Tasks

```sql
-- Create an account role
CREATE ROLE ANALYST;

-- Grant the role to a user
GRANT ROLE ANALYST TO USER Raj;

-- Activate in a session
USE ROLE ANALYST;
```

---

### 2. Database Role

Used to manage permissions **only within a single database**.

```sql
-- Create a database role
CREATE DATABASE ROLE SALES_DB.READER;

-- Grant SELECT privilege to database role
GRANT SELECT ON TABLE SALES_DB.PUBLIC.ORDERS
TO DATABASE ROLE SALES_DB.READER;
```

**Can:**
- ✅ Hold privileges on objects in the same database
- ✅ Be granted to account roles, another database role, or a user

**Cannot:**
- ❌ Be activated using `USE ROLE` command

---

### Database Role Behavior When Granted to a User

When a database role is granted to a user:

1. The privileges held by the database role are **added directly to the user's effective privilege set**
2. The user does **not** need to activate the database role — privileges are available immediately, provided the user is using the relevant database
3. These privileges are independent from the primary/secondary role hierarchy

> **In short:** A database role becomes active in the role hierarchy when the **database containing the database role is in use** or when querying a table in that same database.

---

### Limitation

Authorization to execute `CREATE <object>` statements is provided by the **primary role** only.

If you assign `CREATE` object privilege via a database role to a user, the user will **not** be able to perform the create operation. `CREATE <object>` privileges can only be activated through a primary role.

---

### Example: Assigning Database Role to a User

```sql
-- Create database role
CREATE DATABASE ROLE SALES_DB.READER;

-- Grant permissions to database role
GRANT USAGE ON DATABASE SALES_DB TO DATABASE ROLE SALES_DB.READER;
GRANT USAGE ON SCHEMA SALES_DB.FINANCE_SCHEMA TO DATABASE ROLE SALES_DB.READER;
GRANT SELECT ON TABLE SALES_DB.FINANCE_SCHEMA.SALES TO DATABASE ROLE SALES_DB.READER;
GRANT CREATE TABLE ON SCHEMA SALES_DB.FINANCE_SCHEMA TO DATABASE ROLE SALES_DB.READER;

-- Grant database role to user
GRANT DATABASE ROLE SALES_DB.READER TO USER Raj;
```

> **Note:** Even though `CREATE TABLE` privilege is assigned via the database role, the user still won't be able to create tables because `CREATE <object>` operations can only be performed through a primary role.

---

### Example: Assigning Database Role to an Account Role

```sql
-- Create Account role
CREATE ROLE ANALYST;

-- Create database role
CREATE DATABASE ROLE SALES_DB.READER;

-- Grant permissions to database role
GRANT USAGE ON DATABASE SALES_DB TO DATABASE ROLE SALES_DB.READER;
GRANT USAGE ON SCHEMA SALES_DB.FINANCE_SCHEMA TO DATABASE ROLE SALES_DB.READER;
GRANT SELECT ON TABLE SALES_DB.FINANCE_SCHEMA.SALES TO DATABASE ROLE SALES_DB.READER;
GRANT CREATE TABLE ON SCHEMA SALES_DB.FINANCE_SCHEMA TO DATABASE ROLE SALES_DB.READER;

-- Grant database role to account role
GRANT DATABASE ROLE SALES_DB.READER TO ROLE ANALYST;

-- Grant account role to user
GRANT ROLE ANALYST TO USER Raj;
```

---

### Key Differences

| Aspect | Account Role | Database Role |
|--------|-------------|---------------|
| **Scope** | Account-wide permissions | Database-specific permissions |
| **Assign to users** | ✅ Yes | ✅ Yes |
| **Activate in session** | ✅ `USE ROLE` | ❌ Cannot activate |
| **Grant to** | Users, other account roles | Account roles, database roles, users |
| **CREATE object** | ✅ Supported | ❌ Not effective |

**Summary:** Account Roles are account-level permission containers that can be assigned to users and activated in a session. Database Roles are database-specific permission containers that hold privileges on objects within a single database and can be granted to other Database Roles, Account Roles, and Users.

---

### Securable Objects Hierarchy

![Securable Objects Hierarchy](pasted-image-2026-07-14T00-52-16-217Z.png)

The hierarchy shows how securable objects are organized in Snowflake:
- **Organization** → Account
- **Account** → Warehouse, Database, Role, User, and other account objects
- **Database** → Database Role, Schema
- **Schema** → Table, View, Stage, Stored Procedure, UDF, and other schema objects